# Kunz 2025 · inner speech T12, T15, T16

## Prepare the data

Run the commands below from the repository root with `bci-repro` active.

### 1. Download

Download `sentenceData.zip` from [Dryad](https://doi.org/10.5061/dryad.gf1vhhn1j). Create `studies/20250821_kunz/data/raw/dryad_gf1vhhn1j/` and save the download as `studies/20250821_kunz/data/raw/dryad_gf1vhhn1j/dryad_gf1vhhn1j__sentenceData.zip`.

### 2. Extract

Extract the archive into `studies/20250821_kunz/data/extracted/dryad_gf1vhhn1j/`.

```text
python -m zipfile -e "studies/20250821_kunz/data/raw/dryad_gf1vhhn1j/dryad_gf1vhhn1j__sentenceData.zip" "studies/20250821_kunz/data/extracted/dryad_gf1vhhn1j"
```

After extraction, check for `studies/20250821_kunz/data/extracted/dryad_gf1vhhn1j/sentenceData/`. It contains `sentences_readme.txt` and the per-session `.mat` files.

### 3. Prepare the model input

Convert the released sentence `.mat` files to the HDF5 layout used by the trainer.

```text
python shared/tools/setup_upstream.py --study 20250821_kunz
python studies/20250821_kunz/code/prepare_data.py
```

**Output** `studies/20250821_kunz/data/derived/dryad_gf1vhhn1j/hdf5/trainer_input/`. The converter writes `data_train.hdf5` for open-loop inputs and `data_val.hdf5` for closed-loop evaluation inputs. The session index is `studies/20250821_kunz/data/derived/dryad_gf1vhhn1j/hdf5/dryad_gf1vhhn1j__session_description.csv`.

## Read the archived results

These are the archived mixed-training evaluations. The three arms remain available for supplementary analysis. Earlier Figure 2f OPT weights were selected on the full archived evaluation set.

This walkthrough uses the checked-in archive and takes no data download.

In [1]:
import csv
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "figures/code"))
from figure_paths import PUBLISHED, load_benchmark

benchmark = load_benchmark()

def read_rows(path):
    """Read the archived CSV rows in their original order."""
    with (root / path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

## From a trial or day to the reported result

In [2]:
arm = "Kunz 2025 (inner, T12)"
from build_session_per_vs_wer import build, read_tsv, word_ed

source = benchmark["figure3_per_sources"][arm]
rows = read_rows(source["greedy"])
decoded = read_tsv(root / source["decode"])
trial = rows[0]
reference, hypothesis = decoded[0]
print(f"{trial['session']} · {reference}")
print(f"Word hypothesis: {hypothesis}")
edits, words = word_ed(reference, hypothesis)
print(f"Trial word edits: {edits}/{words}")
result = build(arm, root / source["greedy"], root / source["decode"])
day = result["sessions"][0]
print(f"{day['session']}: {day['n']} trials, PER {day['per']:.2f}%, WER {day['wer']:.2f}%")
print(f"{len(result['sessions'])} sessions, {len(rows)} evaluated trials; pooled {result['pooled']}")

for participant in ("T15", "T16"):
    name = f"Kunz 2025 (inner, {participant})"
    inputs = benchmark["figure3_per_sources"][name]
    summary = build(name, root / inputs["greedy"], root / inputs["decode"])
    print(participant, summary["pooled"])

t12.2023.11.28_imagined_50words_eval · I need success.
Word hypothesis: YOU KNOW WHAT
Trial word edits: 3/3
Kunz 2025 (inner, T12):  1 sessions | pooled PER 61.40% | WER 122.11%
t12.2023.11.28_imagined_50words_eval: 100 trials, PER 61.40%, WER 122.11%
1 sessions, 100 evaluated trials; pooled {'per': 61.4, 'wer': 122.11}
Kunz 2025 (inner, T15):  3 sessions | pooled PER 59.67% | WER 80.86%
T15 {'per': 59.67, 'wer': 80.86}
Kunz 2025 (inner, T16):  2 sessions | pooled PER 61.49% | WER 95.53%
T16 {'per': 61.49, 'wer': 95.53}


## Continue the full reproduction

For the separate T16 block holdout experiment, `code/make_t16_holdout.py` reads the converted HDF5 and writes the fold inputs to `studies/20250821_kunz/data/derived/dryad_gf1vhhn1j/hdf5/t16_holdout/`.

For a separate data disk, set `BCI_DATA_ROOT` to the directory that contains the study folders and use the same layout under that root.

See the [evaluation and input guide](docs/20261124_evaluation_protocol/README.md) and [code guide](code/README.md).

**Public commands** [`prepare_data.py`](code/prepare_data.py), [`train.py`](code/train.py), [`infer.py`](code/infer.py), [`evaluate.py`](code/evaluate.py), [`decode.py`](code/decode.py). Run a command with `--help` to see its arguments.